In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import dense_rank, concat, lit
from pyspark.sql.window import Window
import re

print(f"🗑️ Garantindo que a nova tabela bronze.reservas_raw comece limpa...")
spark.sql(f"DROP TABLE IF EXISTS bronze.reservas_raw")

print("🔄 Lendo a tabela limpa direto do Unity Catalog...")
df_bruto = spark.sql("SELECT * FROM main.default.reservas_anonimizado")

# 1. Criamos a janela baseada na coluna 'Cliente' (com os hashes corrigidos)
janela_anonima = Window.orderBy("Cliente")

print("✨ Substituindo os hashes por pseudônimos amigáveis (Hospede_X)...")
df_anonimo = df_bruto.withColumn(
    "Cliente", 
    concat(lit("Hospede_"), dense_rank().over(janela_anonima))
)

print("🧼 Padronizando as colunas restantes para minúsculas (snake_case)...")
def limpar_nome_coluna(nome):
    nome_limpo = nome.lower().strip()
    substituicoes = {
        "á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u", 
        "ã": "a", "õ": "o", "ç": "c", "â": "a", "ê": "e", "ô": "o"
    }
    for original, substituto in substituicoes.items():
        nome_limpo = nome_limpo.replace(original, substituto)
    nome_limpo = re.sub(r'[ ,;{}()\n\t=\-]', '_', nome_limpo)
    return re.sub(r'_+', '_', nome_limpo).strip('_')

colunas_limpas = [limpar_nome_coluna(c) for c in df_anonimo.columns]
df_final = df_anonimo.toDF(*colunas_limpas)

# 2. Gravação definitiva na camada Bronze do Delta Lake
print(f"💾 Salvando dados finais na tabela bronze.reservas_raw...")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze.reservas_raw")

print(f"✅ Sucesso absoluto! O pipeline criou a tabela bronze.reservas_raw com os dados populados.")

In [0]:
%sql
SELECT * FROM bronze.reservas_raw;